# 06 · Permutation null under dependence and non-stationarity; chronological predictive gain

Responds to the second independent review (GO for NEON feasibility, MODIFY for confirmatory Exp 2a). Agreed estimands (`docs/m0_review2_addendum.md`):

- **E2 = I(D; V | C)** stays the scientific quantity. It is not claimed from any interval here.
- **Operational effect size, G(n_fit):** the expected held-out log-score gain, per row, of the explicitly defined model pair fitted on the first `n_fit` days. The test distribution is the future rows of the same process, restricted to the fit-period support box. Notebook 05 showed G ≠ E2 whenever the context model is imperfect.
- **Zero-information test:** the refitting conditional permutation test, exploratory and subject to its exchangeability assumption. Its p-value is not a confidence bound on E2.

**Pipeline in every replicate.** The contract gate is applied first. Context: hour basis + spline in T + hour × T, plus a **train-only** projection E[V | context]. For each outer fold this is fitted on that fold's training days, out of fold within them, and test rows are predicted. Evaluation and future days never enter it (`inference.project_splits`; this fixes the transductive projection of notebooks 04–05). Scoring uses the robust Gaussian with common support. Tests are one-sided at α = 0.05.

**Null worlds probing the permutation generator.** These are the review's list. True information is 0 in all of them:

| world | stress | scheme |
|---|---|---|
| N0 | none | day units, interleaved day folds |
| NH | visual noise SD varies 4× with T and 1.5× with hour | day |
| NP-day / NP-week | multi-day weather persistence: day levels AR(1) across days, ρ = 0.85, in T, the moisture residual and the visual noise | day units + interleaved day folds / week units + contiguous week folds |
| NR / NR-strata | camera service on day 40: visual offset +1 and noise × 1.6 afterwards | unstratified / strata = before, after |
| NS / NS-strata | 120-day season: T cools 9.6 K, D drifts, the cue has a seasonal hump; context adds a fixed-knot day-of-year spline | unstratified / strata = 30-day periods |
| NM | 15 % of frames missing at random and 20 % of days missing every frame after 13 h | day units, nearest-slot donors |
| `*-shift` | added after the first development run, which showed the persistence failure: **cyclic-shift null** (the V-residual series is shifted by every k ≥ 7 whole days, preserving its serial dependence) with contiguous 16-day folds and a 3-day purge | shift |

Positives: PP-day / PP-week (persistence, truth 0.05) and PM (missing frames, truth 0.05). Development uses 100 replicates, 39 permutations and seeds 8000+.

In [1]:
from dataclasses import replace

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from visual_hygrometry import data, inference, policy, scoring, splits, synthetic, worlds
from visual_hygrometry.worlds import frame, gauss_world, iid

pd.set_option('display.precision', 4)
par = lambda: Parallel(n_jobs=16, max_nbytes=None)
KNOTS, DOY0, DOY_KNOTS = np.arange(-4, 45, 4.0), 150, np.arange(130, 311, 20.0)  # fixed before any run


def design(df, season):
    df, ctx = worlds.add_context_basis(df, KNOTS)
    raw = ['hour_local', 'temp_cond_c']
    if season:
        df['doy'] = DOY0 + df.day
        S = data.spline(df.doy, DOY_KNOTS, 'doy_spl')
        df, ctx, raw = pd.concat([df, S], axis=1), ctx + list(S), raw + ['doy']
    return df, ctx, raw


def one_rep(W, seed, n_perm=39, n_pop=300):
    n = W['n_days']
    df, ctx, rawc = design(frame(W, n + n_pop, seed), W.get('season', False))
    Xc, Xvc, y, _ = policy.select_sci(df, 'e2', ctx, W['vis'])
    Cb, V, y = Xc.to_numpy(), Xvc.to_numpy()[:, Xc.shape[1]:], y.to_numpy()
    raw, g, h = df[rawc].to_numpy(), df.day.to_numpy(), df.hour.to_numpy()
    s, blk, scheme = g < n, g // 7, W.get('units', 'day')
    if scheme == 'shift':  # contiguous 16-day folds, training days within 3 days of a test day purged
        t = pd.Series(pd.to_datetime(g[s], unit='D'))
        sp = splits.kfold(splits.assign_folds(g[s] // 16, 5, 'contiguous'), t, t, embargo=f"{W.get('purge', 3)}D")
    else:
        sp = splits.kfold(splits.assign_folds(blk[s], 5, 'contiguous') if scheme == 'week' else splits.assign_folds(g[s], 5))
    C = [np.c_[Cb[s], p] for p in inference.project_splits(raw[s], V[s], sp, g[s])]
    keep = scoring.support_mask(raw[s][:, :2], sp)
    r = scoring.contrast(C, [np.c_[c, V[s]] for c in C], y[s], sp, g[s], keep=keep, robust=True)
    Cnull = np.c_[Cb[s], inference.project(raw[s], V[s], g[s])]  # null generator: (C, V) of the analysis sample, never D
    units, slot = (blk[s], (g[s] % 7) * 24 + h[s]) if scheme == 'week' else (g[s], h[s])
    strata = W['strata'](df[s]) if 'strata' in W else None
    _, p, _ = inference.perm_test(C, V[s], y[s], sp, g[s], slot, n_perm, seed + 500_000, keep=keep, units=units, strata=strata, Cnull=Cnull,
                                  shift_min=W.get('shift_min', 7) if scheme == 'shift' else None, robust=True)
    fit, ev = np.zeros(len(g), bool), np.zeros(len(g), bool)
    f, e = splits.chrono(g[s], 0.6, gap=2)
    fit[np.flatnonzero(s)[f]], ev[np.flatnonzero(s)[e]] = True, True
    spc = [(fit, ev | ~s)]
    Pc = inference.project_splits(raw, V, spc, g)[0]
    rc = scoring.contrast(np.c_[Cb, Pc], np.c_[Cb, Pc, V], y, spc, g, keep=scoring.support_mask(raw[:, :2], spc), robust=True)
    dz = np.full(len(g), np.nan)
    dz[rc.scored] = rc.d
    e = ev & np.isfinite(dz)
    dl, dh = inference.cluster_interval(dz[e], g[e])
    wl, wh = inference.cluster_interval(dz[e], blk[e])
    hl, hh = inference.hac_interval(dz[e], g[e])
    return dict(est=r.mean, p=p, chrono=dz[e].mean(), day_lo=dl, day_hi=dh, wk_lo=wl, wk_hi=wh, hac_lo=hl, hac_hi=hh, target=np.nanmean(dz[~s]), excluded=1 - keep.mean())


def run(W, seeds, n_perm=39): return pd.DataFrame(par()(delayed(one_rep)(W, s, n_perm) for s in seeds))


def summarise(name, W, a):
    t, stationary = W['truth'], not W.get('season', False)
    cov = lambda lo, hi: ((a[lo] <= a.target) & (a[hi] >= a.target)).mean() if stationary else np.nan
    return dict(world=name, truth=t, reps=len(a), rej_perm=(a.p <= .05).mean(), bias=a.est.mean() - t,
                target_minus_truth=a.target.mean() - t if stationary else np.nan, chrono_minus_target=(a.chrono - a.target).mean() if stationary else np.nan,
                cov_day=cov('day_lo', 'day_hi'), cov_week=cov('wk_lo', 'wk_hi'), cov_hac=cov('hac_lo', 'hac_hi'), excluded=a.excluded.mean())

In [2]:
def gw(gamma=0.0, persist=0.0, **kw):
    spec = gauss_world(gamma, t_sd=4.0, **kw)
    return {k: replace(n, persist=persist) if persist and n.day > 0 else n for k, n in spec.items()}

def hetero(df, rng):
    T, h = df.temp_cond_c.to_numpy(), df.hour.to_numpy()
    df['vis_moist'] = df.vis_moist * (0.4 + 0.15 * np.abs(T - 20)) * (1 + 0.5 * np.cos(2 * np.pi * (h - 13) / 24))
    return df

def camera_service(df, rng):
    after = df.day >= 40
    df.loc[after, 'vis_moist'] = 1.6 * df.loc[after, 'vis_moist'] + 1.0
    return df

def season(df, rng):
    t = df.day.to_numpy()
    df['temp_cond_c'] = df.temp_cond_c - 0.08 * t
    df['d_ref_k'] = df.d_ref_k - 0.03 * t
    df['vis_moist'] = df.vis_moist + 1.5 * np.sin(np.pi * np.clip(t, 0, 120) / 120)
    return df

def missing(df, rng):
    pm = rng.random(df.day.max() + 1)[df.day] < .2
    return df[~((rng.random(len(df)) < .15) | (pm & (df.hour >= 13)))].reset_index(drop=True)

persist_kw = dict(persist=.85, r_day=.7, v_day=.6)
truth = lambda spec: synthetic.cmi(spec, 'd_ref_k', 'vis_moist', 'temp_cond_c')
DEV = {'N0': dict(spec=gw()), 'NH': dict(spec=gw(), modify=hetero),
       'NP-day': dict(spec=gw(**persist_kw)), 'NP-week': dict(spec=gw(**persist_kw), units='week'),
       'NR': dict(spec=gw(), modify=camera_service), 'NR-strata': dict(spec=gw(), modify=camera_service, strata=lambda d: (d.day >= 40).to_numpy()),
       'NS': dict(spec=gw(), modify=season, season=True, n_days=120), 'NS-strata': dict(spec=gw(), modify=season, season=True, n_days=120, strata=lambda d: (d.day // 30).to_numpy()),
       'NM': dict(spec=gw(), modify=missing),
       'PP-day': dict(spec=gw(.162, **persist_kw)), 'PP-week': dict(spec=gw(.162, **persist_kw), units='week'), 'PM': dict(spec=gw(.162), modify=missing),
       # added after the first development run (persistence failure): cyclic-shift null with contiguous purged folds
       'N0-shift': dict(spec=gw(), units='shift'), 'NP-shift': dict(spec=gw(**persist_kw), units='shift'), 'NR-shift': dict(spec=gw(), modify=camera_service, units='shift'),
       'NS-shift': dict(spec=gw(), modify=season, season=True, n_days=120, units='shift'), 'NM-shift': dict(spec=gw(), modify=missing, units='shift'),
       'PP-shift': dict(spec=gw(.162, **persist_kw), units='shift')}
DEV = {k: {'n_days': 80, 'vis': ['vis_moist']} | W | {'truth': truth(W['spec'])} for k, W in DEV.items()}
dev_raw = {k: run(W, range(8000, 8100)) for k, W in DEV.items()}
dev = pd.DataFrame([summarise(k, DEV[k], a) for k, a in dev_raw.items()]).set_index('world')
dev

,truth,reps,rej_perm,bias,target_minus_truth,chrono_minus_target,cov_day,cov_week,cov_hac,excluded
world,,,,,,,,,,
N0,0.0000,100,0.07,-0.0014,-0.0018,-1.2663e-04,0.97,0.95,0.94,0.0150
NH,0.0000,100,0.07,-0.0014,-0.0014,4.0042e-05,0.94,0.93,0.89,0.0150
NP-day,0.0000,100,0.38,0.0124,-0.0184,6.8459e-03,0.58,0.83,0.65,0.0132
NP-week,0.0000,100,0.14,-0.0035,-0.0184,6.8459e-03,0.58,0.83,0.65,0.0224
NR,0.0000,100,0.09,-0.0014,-0.0053,-7.0304e-04,0.93,0.94,0.91,0.0150
NR-strata,0.0000,100,0.07,-0.0014,-0.0053,-7.0304e-04,0.93,0.94,0.91,0.0150
NS,0.0000,100,0.07,-0.0008,NaN,NaN,NaN,NaN,NaN,0.0134
NS-strata,0.0000,100,0.06,-0.0008,NaN,NaN,NaN,NaN,NaN,0.0134
NM,0.0000,100,0.05,-0.0015,-0.0017,-1.7509e-04,0.94,0.97,0.92,0.0154


## 2 · Persistence sweep

The first two development runs showed that multi-day persistence is the one stress the permutation schemes do not absorb (ρ = 0.85: day units 0.38, week units 0.14, cyclic shift 0.12). This sweep maps Type I error against the day-to-day persistence ρ of the weather and visual-noise levels, for three schemes:
- `day`: day units, interleaved folds;
- `shift`: cyclic shift ≥ 7 days, contiguous 16-day folds, 3-day purge;
- `shift7`: cyclic shift ≥ 10 days, 7-day purge.

Nulls, 100 replicates, 80 days.

In [3]:
SWEEP = {f'{scheme} rho={rho}': {'n_days': 80, 'vis': ['vis_moist'], 'spec': gw(**(dict(persist=rho, r_day=.7, v_day=.6) if rho else {})), 'truth': 0.0}
         | ({'units': 'shift'} if scheme != 'day' else {}) | ({'purge': 7, 'shift_min': 10} if scheme == 'shift7' else {})
         for scheme in ['day', 'shift', 'shift7'] for rho in [0.0, 0.3, 0.6, 0.85]}
sweep = pd.DataFrame([summarise(k, W, run(W, range(8200, 8300))) for k, W in SWEEP.items()]).set_index('world')
sweep[['rej_perm', 'bias', 'cov_day', 'cov_week', 'cov_hac']]

,rej_perm,bias,cov_day,cov_week,cov_hac
world,,,,,
day rho=0.0,0.07,-0.0015,0.92,0.94,0.90
day rho=0.3,0.07,-0.0022,0.93,0.93,0.90
day rho=0.6,0.18,0.0004,0.84,0.96,0.86
day rho=0.85,0.38,0.0088,0.51,0.85,0.59
shift rho=0.0,0.05,-0.0017,0.92,0.94,0.90
shift rho=0.3,0.06,-0.0042,0.93,0.93,0.90
shift rho=0.6,0.06,-0.0063,0.84,0.96,0.86
shift rho=0.85,0.10,-0.0090,0.51,0.85,0.59
shift7 rho=0.0,0.04,-0.0015,0.92,0.94,0.90


## 3 · Reading the development suite

*Written after the development runs, before the validation run.*

- **Absorbed by day-unit permutation** (Type I within MC error, SE ≈ 0.022): heteroscedastic visual noise (0.07), missing frames with nearest-slot donors (0.05), seasonal drift with a day-of-year context (0.07; 0.06 stratified) and a camera service break (0.09; 0.07 when stratified by the known break).
- **Not absorbed: multi-day persistence.** Day units give 0.07 / 0.07 / 0.18 / 0.38 at ρ = 0 / 0.3 / 0.6 / 0.85. Interleaved day folds also bias the null estimate upward (+0.012 at ρ = 0.85). Neighbouring training and test days share weather states, so chance correlations learned in training "validate" on their neighbours.
- **Cyclic-shift null** (V-residual series shifted by every k ≥ 7 days, contiguous 16-day folds, 3-day purge): 0.05 / 0.06 / 0.06 / 0.10 over the same ρ. It is valid up to ρ = 0.6 and marginal at 0.85. A 7-day purge does not help (0.13). Costs: power at ρ = 0.85 falls to 0.47, and contiguous folds extrapolate across a season (bias −0.027, conservative).
- **Chronological hold-out for G:** week-cluster t intervals cover the target 0.93–0.96 for ρ ≤ 0.6 and 0.85 at ρ = 0.85. Day clusters (0.51 at 0.85) and Newey–West (0.59) fail, because 30 evaluation days cannot estimate a long-run variance. Fitting on 48 of 80 days captures less than E2 (target − truth −0.002 without persistence, −0.018 to −0.020 at ρ = 0.85), as the agreed estimand definition anticipates.
- **Real-data anchor:** daytime daily-mean D at HARV in July 2023 has lag-1 autocorrelation +0.03 (notebook 07). That is one month only; persistence must be estimated per site and season before a scheme is chosen.

**Chosen for validation:**
- Q0: cyclic-shift permutation, stratified at declared regime breaks.
- G: chronological hold-out (60 % / 2-day purge / 40 %) with week-cluster t intervals.
- Declared validity region: ρ ≤ 0.6.
- Day units are run alongside for reference.

## 4 · Support threshold and robust-score sensitivity

**Support.** In the worlds above the information per row does not depend on the context, so restricting scoring to the support leaves the truth unchanged. Two worlds here make the information depend strongly on temperature:
- **cold end**: the cue carries the moisture residual only below 14 °C, a fog-like case at the cold tail;
- **warm end**: the cue's gain rises linearly above 18 °C.

The per-row truth is exact, ½ log(1 + g(T)² σ_r² / τ²). The truth on the support box (training quantiles q … 1 − q of hour and T) is computed by Monte Carlo on an iid copy of each world. The joint model gets V and V × (T − 20)/5 (`visx_temp`), so its visual coefficient can vary linearly with T. 60 replicates, 80 days.

**Robust score.** The defensive mixture's log loss is *not bounded*; it only grows logarithmically. Its weight ε and Cauchy scale both change the operational predictive and therefore G, so they are varied on a plain world and a tail-prone world (exponential context term encoded by the cue, T anomaly SD 4.5 K, as in notebook 05).

In [4]:
def info_world(kind):
    def f(df, rng):
        T = df.temp_cond_c.to_numpy()
        gain = np.where(T < 14, 0.6, 0.0) if kind == 'cold' else 0.12 * np.maximum(T - 18, 0)
        df['vis_moist'] = df.vis_moist + gain * df.dres
        df['row_info'] = 0.5 * np.log1p(gain ** 2 * 4.0 / 1.0)
        df['visx_temp'] = df.vis_moist * (T - 20) / 5
        return df
    return dict(spec=gw(), vis=['vis_moist', 'visx_temp'], modify=f, n_days=80)

QS = [0.0, 0.001, 0.005, 0.025]

def support_truth(W, n_days=8000):
    df, ctx, raw = design(frame(iid(W), n_days, 1), False)
    R = df[['hour_local', 'temp_cond_c']].to_numpy()
    out = {}
    for q in QS:
        lo, hi = np.quantile(R, [q, 1 - q], axis=0)
        inside = np.all((R >= lo) & (R <= hi), axis=1)
        out[q] = dict(truth=df.row_info[inside].mean(), excluded=1 - inside.mean(), mean_T_excluded=df.temp_cond_c[~inside].mean() if (~inside).any() else np.nan)
    return out

def sens_rep(W, seed, settings):
    df, ctx, rawc = design(frame(W, W['n_days'], seed), False)
    Xc, Xvc, y, _ = policy.select_sci(df, 'e2', ctx, W['vis'])
    Cb, V, y = Xc.to_numpy(), Xvc.to_numpy()[:, Xc.shape[1]:], y.to_numpy()
    raw, g = df[rawc].to_numpy(), df.day.to_numpy()
    sp = splits.kfold(splits.assign_folds(g, 5))
    C = [np.c_[Cb, p] for p in inference.project_splits(raw, V, sp, g)]
    out = {}
    for name, (q, robust) in settings.items():
        r = scoring.contrast(C, [np.c_[c, V] for c in C], y, sp, g, keep=scoring.support_mask(raw, sp, q) if q else None, robust=robust)
        out |= {name: r.mean, f'{name}:worst': r.d.min()}
    return out

S_SUPPORT = {f'q={q}': (q, True) for q in QS}
rows = []
for kind in ['cold', 'warm']:
    W = info_world(kind)
    st = support_truth(W)
    a = pd.DataFrame(par()(delayed(sens_rep)(W, s, S_SUPPORT) for s in range(9000, 9060)))
    for q in QS:
        rows.append(dict(world=kind, q=q, excluded=st[q]['excluded'], mean_T_excluded=st[q]['mean_T_excluded'], E2_on_support=st[q]['truth'],
                         E2_full=st[0.0]['truth'], estimate=a[f'q={q}'].mean(), sd=a[f'q={q}'].std(), worst_row=a[f'q={q}:worst'].min()))
pd.DataFrame(rows).set_index(['world', 'q'])

excluded  mean_T_excluded  E2_on_support  E2_full  estimate  \
world q                                                                    
cold  0.000    0.0000              NaN         0.0231   0.0231    0.0060   
      0.001    0.0020          21.5529         0.0227   0.0231    0.0052   
      0.005    0.0100          21.5871         0.0210   0.0231    0.0046   
      0.025    0.0879          21.3084         0.0118   0.0231    0.0009   
warm  0.000    0.0000              NaN         0.4233   0.4233    0.2489   
      0.001    0.0020          21.5529         0.4226   0.4233    0.2506   
      0.005    0.0100          21.5871         0.4203   0.4233    0.2521   
      0.025    0.0879          21.3084         0.4151   0.4233    0.2580   

                 sd  worst_row  
world q                         
cold  0.000  0.0086    -2.7105  
      0.001  0.0075    -2.4252  
      0.005  0.0071    -2.0890  
      0.025  0.0054    -2.0890  
warm  0.000  0.0510    -7.5255  
      0.001  0.0513    -5.3136  
      0.005  0.0511    -5.3136  
      0.025  0.0533    -4.9608

In [5]:
expo = lambda T: 0.6 * np.exp((T - 20) / 5)
def tail(df, rng):
    T = df.temp_cond_c.to_numpy()
    df['d_ref_k'] = df.d_ref_k + expo(T)
    df['vis_moist'] = expo(T) / 0.6 + np.sqrt(np.expm1(.2)) * .3 / 2 * df.dres + rng.normal(0, .3, len(df))
    return df
R_WORLDS = {'plain': dict(spec=gw(.162), vis=['vis_moist'], n_days=80, truth=truth(gw(.162))),
            'tail': dict(spec=gauss_world(0.0, t_sd=4.5), vis=['vis_moist'], n_days=80, modify=tail, truth=0.10)}
S_ROBUST = {'Gaussian': (0.005, False), 'eps 0.5% scale 4': (0.005, (0.005, 4.0)), 'eps 1% scale 2': (0.005, (0.01, 2.0)), 'eps 1% scale 4 (chosen)': (0.005, True),
            'eps 1% scale 8': (0.005, (0.01, 8.0)), 'eps 2% scale 4': (0.005, (0.02, 4.0))}
rows = []
for k, W in R_WORLDS.items():
    a = pd.DataFrame(par()(delayed(sens_rep)(W, s, S_ROBUST) for s in range(9100, 9160)))
    for name in S_ROBUST:
        rows.append(dict(world=k, setting=name, truth_E2=W['truth'], mean=a[name].mean(), median=a[name].median(), sd=a[name].std(), worst_row=a[f'{name}:worst'].min()))
pd.DataFrame(rows).set_index(['world', 'setting'])

truth_E2    mean  median      sd  worst_row
world setting                                                             
plain Gaussian                   0.0499  0.0453  0.0424  0.0202    -3.0599
      eps 0.5% scale 4           0.0499  0.0453  0.0427  0.0201    -2.8539
      eps 1% scale 2             0.0499  0.0453  0.0428  0.0201    -2.8237
      eps 1% scale 4 (chosen)    0.0499  0.0453  0.0429  0.0201    -2.8212
      eps 1% scale 8             0.0499  0.0453  0.0428  0.0201    -2.8421
      eps 2% scale 4             0.0499  0.0452  0.0431  0.0201    -2.7583
tail  Gaussian                   0.1000  0.0932  0.0924  0.0159    -4.8297
      eps 0.5% scale 4           0.1000  0.0933  0.0925  0.0160    -4.0757
      eps 1% scale 2             0.1000  0.0934  0.0924  0.0160    -3.9811
      eps 1% scale 4 (chosen)    0.1000  0.0934  0.0924  0.0160    -3.9547
      eps 1% scale 8             0.1000  0.0934  0.0925  0.0160    -4.0189
      eps 2% scale 4             0.1000  0.0934  0.0922  0.0160    -3.7481

- **Support.** The box excludes 0.2 / 1.0 / 8.8 % of rows at q = 0.1 / 0.5 / 2.5 %, mostly the first and last time slots plus temperature extremes. When information lives at the cold tail, the support-restricted E2 falls from 0.023 to 0.021 at q = 0.5 % and is halved (0.012) at q = 2.5 %. Trimming can remove the regime that carries the information. q = 0.5 % is kept, the excluded share is reported, and support-dependent truth must be checked in every new world.
- **Operational vs fundamental estimand.** With information varying in T, even the joint model with V and V × T captures 25 % (cold end) and 60 % (warm end) of E2. G can be far below E2 when the visual model is too simple. A positive G with a correct context model is evidence of information; a small G is not evidence of small E2.
- **Robust score.** Mean contrast is identical to four decimals across ε = 0.5–2 %, Cauchy scale 2–8 σ and plain Gaussian scoring (plain world 0.0453, tail world 0.0932–0.0934; worst rows −2.8 to −4.8 nats). With the support restriction in place, the mixture is a secondary safeguard and the 1 % / 4σ choice is not influential here. Its log loss is still unbounded, only less sensitive.

## 5 · Fresh validation suite

Worlds not used before: 36 samples/day over 06:30–18:30, 90 days, β = 0.5, T anomaly SD 3.6 K, moisture residual SD 1.7 K; a camera service on day 55 (offset −1.5, noise × 0.7) with strata at the declared break; a 90-day season; 10 % random missing frames plus missing mornings on 25 % of days; a hinge mixed cue; seeds 130 000+. 150 replicates. The shift null enumerates every allowed shift; the day-unit reference uses 199 permutations.

**Criteria, declared before the first validation run**, for the chosen procedure (cyclic shift):

| id | criterion |
|---|---|
| V1 | permutation Type I ≤ 0.085 in each null Y-N1 … Y-N6 (Y-N3 has ρ = 0.3) |
| V2 | power ≥ 0.80 in Y-P1 (plain, 0.05 nats) and Y-P3 (mixed hinge cue, 0.05) |
| V3 | chronological week-cluster interval covers its target G ≥ 0.90 in every stationary world (all except Y-N5) |
| V4 | mean chronological estimate within 0.005 of its target G in those worlds |
| V5 | cross-fitted estimate within 0.01 of E2 in Y-P1 and Y-P2 |
| V6 | off-support share ≤ 0.05 |
| — | Y-X (ρ = 0.6, edge of the declared region): reported, no criterion |

In [6]:
VQ = dict(per_day=36, hours=(6.5, 18.5), n_days=90)
qw = lambda gamma=0.0, persist=0.0: {k: replace(n, persist=persist) if persist and n.day > 0 else n
                                     for k, n in gauss_world(gamma, beta=.5, t_sd=3.6, r_sd=1.7, r_day=.6, r_ar=.85, v_day=.4, v_ar=.6).items()}
gq = lambda c, v_sd=1.0: np.sqrt(np.expm1(2 * c)) * v_sd / 1.7

def hetero_q(df, rng):
    T, h = df.temp_cond_c.to_numpy(), df.hour.to_numpy()
    df['vis_moist'] = df.vis_moist * (0.3 + 0.2 * np.abs(T - 21)) * (1 + 0.6 * np.sin(2 * np.pi * (h - 9) / 24))
    return df

def service_q(df, rng):
    after = df.day >= 55
    df.loc[after, 'vis_moist'] = 0.7 * df.loc[after, 'vis_moist'] - 1.5
    return df

def season_q(df, rng):
    t = df.day.to_numpy()
    df['temp_cond_c'] = df.temp_cond_c + 0.06 * t
    df['d_ref_k'] = df.d_ref_k + 0.02 * t
    df['vis_moist'] = df.vis_moist - 1.2 * np.cos(np.pi * np.clip(t, 0, 90) / 90)
    return df

def missing_q(df, rng):
    morning = rng.random(df.day.max() + 1)[df.day] < .25
    return df[~((rng.random(len(df)) < .1) | (morning & (df.hour < 9.5)))].reset_index(drop=True)

def hinge_q(df, rng):
    T = df.temp_cond_c.to_numpy()
    df['d_ref_k'] = df.d_ref_k + 0.6 * np.maximum(T - 23, 0)
    df['vis_moist'] = np.maximum(T - 23, 0) + gq(.05, .3) * df.dres + rng.normal(0, .3, len(df))
    return df

VAL = {'Y-N1': dict(spec=qw()), 'Y-N2': dict(spec=qw(), modify=hetero_q), 'Y-N3': dict(spec=qw(persist=.3)),
       'Y-N4': dict(spec=qw(), modify=service_q, strata=lambda d: (d.day >= 55).to_numpy()),
       'Y-N5': dict(spec=qw(), modify=season_q, season=True), 'Y-N6': dict(spec=qw(), modify=missing_q),
       'Y-P1': dict(spec=qw(gq(.05))), 'Y-P2': dict(spec=qw(gq(.05)), modify=missing_q), 'Y-P3': dict(spec=qw(), modify=hinge_q, truth=0.05),
       'Y-X': dict(spec=qw(persist=.6))}
VAL = {f'{k}|{sch}': VQ | {'vis': ['vis_moist'], 'units': sch} | W | {'truth': W.get('truth', truth(W['spec']))} for k, W in VAL.items() for sch in ('shift', 'day')}
val_raw = {k: run(W, range(130000, 130150), n_perm=199) for k, W in VAL.items()}
val = pd.DataFrame([summarise(k, VAL[k], a) for k, a in val_raw.items()]).set_index('world')
val

,truth,reps,rej_perm,bias,target_minus_truth,chrono_minus_target,cov_day,cov_week,cov_hac,excluded
world,,,,,,,,,,
Y-N1|shift,0.00,150,0.0600,-0.0020,-0.0026,1.0557e-03,0.9133,0.9267,0.8733,0.0154
Y-N1|day,0.00,150,0.0600,-0.0017,-0.0026,1.0557e-03,0.9133,0.9267,0.8733,0.0141
Y-N2|shift,0.00,150,0.0333,-0.0017,-0.0025,9.8591e-04,0.8933,0.9133,0.8933,0.0154
Y-N2|day,0.00,150,0.0400,-0.0013,-0.0025,9.8591e-04,0.8933,0.9133,0.8933,0.0141
Y-N3|shift,0.00,150,0.0600,-0.0021,-0.0029,8.4054e-04,0.9067,0.9133,0.8800,0.0165
Y-N3|day,0.00,150,0.1000,-0.0013,-0.0029,8.4054e-04,0.9067,0.9133,0.8800,0.0141
Y-N4|shift,0.00,150,0.0467,-0.0055,-0.0056,5.4458e-04,0.9400,0.9533,0.9400,0.0154
Y-N4|day,0.00,150,0.0733,-0.0029,-0.0056,5.4458e-04,0.9400,0.9533,0.9400,0.0141
Y-N5|shift,0.00,150,0.0667,-0.0132,NaN,NaN,NaN,NaN,NaN,0.0229


In [7]:
v = lambda w, c: val.loc[f'{w}|shift', c]
nulls, stationary = [f'Y-N{i}' for i in range(1, 7)], ['Y-N1', 'Y-N2', 'Y-N3', 'Y-N4', 'Y-N6', 'Y-P1', 'Y-P2', 'Y-P3']
acc = [('V1', 'shift Type I <= 0.085, Y-N1..N6', [v(w, 'rej_perm') for w in nulls], all(v(w, 'rej_perm') <= .085 for w in nulls)),
       ('V2', 'power >= 0.80, Y-P1 / Y-P3', [v(w, 'rej_perm') for w in ('Y-P1', 'Y-P3')], all(v(w, 'rej_perm') >= .8 for w in ('Y-P1', 'Y-P3'))),
       ('V3', 'week-cluster interval covers G >= 0.90', [v(w, 'cov_week') for w in stationary], all(v(w, 'cov_week') >= .9 for w in stationary)),
       ('V4', '|chrono - G| <= 0.005', [v(w, 'chrono_minus_target') for w in stationary], all(abs(v(w, 'chrono_minus_target')) <= .005 for w in stationary)),
       ('V5', '|cross-fit - E2| <= 0.01, Y-P1 / Y-P2', [v(w, 'bias') for w in ('Y-P1', 'Y-P2')], all(abs(v(w, 'bias')) <= .01 for w in ('Y-P1', 'Y-P2'))),
       ('V6', 'off-support share <= 0.05', val.excluded.max(), val.excluded.max() <= .05)]
display(val.loc[[f'Y-X|{sch}' for sch in ('shift', 'day')], ['rej_perm', 'cov_week']])
pd.DataFrame([(i, c, np.round(x, 3), ok) for i, c, x, ok in acc], columns=['id', 'criterion', 'value', 'pass'])

,rej_perm,cov_week
world,,
Y-X|shift,0.0467,0.9
Y-X|day,0.1467,0.9


,id,criterion,value,pass
0,V1,"shift Type I <= 0.085, Y-N1..N6","[0.06, 0.033, 0.06, 0.047, 0.067, 0.053]",True
1,V2,"power >= 0.80, Y-P1 / Y-P3","[0.993, 1.0]",True
2,V3,week-cluster interval covers G >= 0.90,"[0.927, 0.913, 0.913, 0.953, 0.933, 0.96, 0.94...",True
3,V4,|chrono - G| <= 0.005,"[0.001, 0.001, 0.001, 0.001, 0.001, 0.0, 0.002...",True
4,V5,"|cross-fit - E2| <= 0.01, Y-P1 / Y-P2","[-0.001, -0.001]",True
5,V6,off-support share <= 0.05,0.023,True


**Validation outcome: all six criteria pass** on worlds not used in development.

- **Q0:** the cyclic-shift test holds Type I at 0.033–0.067 in six nulls (heteroscedastic visual noise, ρ = 0.3 persistence, a camera service stratified at its declared break, a 90-day season, missing frames and mornings). Power at 0.05 nats is 0.99–1.00, including a cue that also encodes a hinge in temperature. At ρ = 0.6, the edge of the declared region, it holds 0.047.
- **Day-unit reference:** 0.10 at ρ = 0.3 and 0.147 at ρ = 0.6, which confirms the choice of the shift null.
- **G:** the chronological week-cluster interval covers its target in 0.913–0.960 of replicates; mean chronological estimates are within 0.002 of G, and the cross-fitted estimate is within 0.001 of E2 on plain cues. Off-support share is ≤ 2.3 %.

**Limits of what passed.**
- **Synthetic only.** Validity is shown for ρ ≤ 0.6, not beyond (0.10 at ρ = 0.85 in development).
- **Seasons.** Contiguous folds extrapolate across a season (cross-fitted bias −0.013 in Y-N5, conservative).
- **Test resolution.** The shift p-value resolution is 1/(G − 13) ≈ 0.013 at 90 days.
- **Estimand.** G is the information captured by a half-record model and is systematically below E2 under persistence or model inadequacy.
- **Assumption.** The test's exchangeability assumption (stationary residual series within strata) is untestable from one record and must be argued per site.